# Dunnhumby K=1 M4 개선 screen (6 arm)
원 논문 BPR(양성당 음성 1개)을 모든 arm에서 유지합니다. hard-negative는 음성이 2개 이상 필요하므로 제외했습니다. M1·M2와 세 가지 M4(원형 / 첫 구매 행 한정 / 보완 가중)의 M4·M5 쌍을 같은 실행에서 학습합니다. seed 42, DAY 1~683 학습 → 684~690 개발평가, 100 epoch, 8회 학습(약 4시간 40분). 끊기면 체크포인트에서 자동 재개합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '20840d37ed4ff5412210bf849278b5c44e605fb5'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA


In [ ]:
import json
import torch
import lightgcn_clv_m5_k1_m4_improvement_screen as improvement

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
assert improvement.CODE_VERSION == 'm5-k1-m4-improvement-screen-v1'
cfg = improvement.configure_improvement_screen()
summary = improvement.preflight_summary(cfg)
assert summary['loss']['negative_count'] == 1
assert summary['loss']['hard_negative'] is False
assert summary['m2']['rho'] == 0.25
assert summary['reused_models'] == []
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = improvement.run_improvement_screen(cfg)


In [ ]:
from IPython.display import display

def show(frame):
    view = frame.copy()
    view.attrs = {}
    display(view)

core = ['model_id', 'improvement', 'rho', 'row_weight_cv', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
        'recall@50', 'ndcg@50', 'price_purchase_amount_weighted_hit@10', 'vndcg@10',
        'user_value_tendency_recommended_price_alignment']
print('1) 여섯 모형 핵심 절대지표')
show(result_df[[column for column in core if column in result_df.columns]])
print('2) Top-10 목록 변경 비율 (지표 해석 전 먼저 확인)')
show(result_df.attrs['top10_overlap'])
print('3) 판독')
print(json.dumps(result_df.attrs['decision'], ensure_ascii=False, indent=2))
print('저장 파일:', json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
